In [ ]:
# === 0. BATCH INGESTION (10 SNOOP TRACKS) ===
import os
import subprocess
import time

print("🎧 Starting Batch Ingestion of 10 Snoop Tracks...")
# A playlist or 10 specific acapella tracks (Using a generic search query for 10 items)
download_cmd = [
    "yt-dlp", 
    "ytsearch10:snoop dogg acapella vocals only", 
    "--extract-audio", 
    "--audio-format", "wav", 
    "-o", "snoop_train_%(autonumber)s.%(ext)s"
]

try:
    subprocess.run(download_cmd, check=True)
    print("✅ Successfully ripped 10 pure audio tracks for the Timbral Optimizer!")
except Exception as e:
    print(f"⚠️ Batch download failed: {e}")


In [ ]:
# === 1. SOVEREIGN AUDIO MATH (YT-DLP INGESTION) ===
import os
import subprocess
import librosa
import soundfile as sf
import numpy as np
import torch
import IPython.display as ipd
import warnings
warnings.filterwarnings('ignore')
import sys

def download_clip(query, output_name, duration="00:00:15"):
    print(f"Downloading {output_name} via search: '{query}'...")
    cmd = [
        sys.executable, "-m", "yt_dlp",
        "-f", "bestaudio[ext=m4a]",
        "--match-filter", "!is_live",
        "--download-sections", "*00:01:30-00:01:45",
        "-o", output_name + ".m4a",
        f"ytsearch1:{query}"
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for f in ["snoop1.m4a", "snoop2.m4a", "snoop3.m4a", "instructor_input.m4a"]:
    if os.path.exists(f): os.remove(f)

# Download 3 Snoop Dogg Interviews (Raw Vocals)
download_clip("snoop dogg breakfast club interview", "snoop1")
download_clip("snoop dogg howard stern interview", "snoop2")
download_clip("snoop dogg jimmy kimmel interview", "snoop3")

# Download 1 Medical Instructor (Input)
download_clip("medical instructor cardiovascular lecture", "instructor_input")

print("✅ Data Ingestion Complete!")


In [ ]:
# === 2. EXTRACTING THE DSP TRUTH MATRIX ===
def extract_dsp_footprint(filepath):
    if os.path.exists(filepath + ".m4a"):
        y, sr = librosa.load(filepath + ".m4a", sr=22050)
    elif os.path.exists(filepath + ".wav"):
        y, sr = librosa.load(filepath + ".wav", sr=22050)
    else:
        return 100.0, np.zeros(13), np.zeros(1024), 22050
    
    f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    valid_f0 = f0[voiced_flag]
    mean_pitch = np.median(valid_f0) if len(valid_f0) > 0 else 100.0
    
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mean_mfccs = np.mean(mfccs, axis=1)
    return mean_pitch, mean_mfccs, y, sr

snoop_pitches = []
snoop_mfccs_list = []
for f in ["snoop1", "snoop2", "snoop3"]:
    p, m, _, _ = extract_dsp_footprint(f)
    snoop_pitches.append(p)
    snoop_mfccs_list.append(m)

snoop_pitch = np.mean(snoop_pitches)
snoop_mfcc = np.mean(snoop_mfccs_list, axis=0)

instructor_pitch, instructor_mfcc, instructor_y, sr = extract_dsp_footprint("instructor_input")

print(f"🎤 Instructor Pitch: {instructor_pitch:.1f} Hz")
print(f"🌿 Snoop Averaged Pitch: {snoop_pitch:.1f} Hz")

pitch_shift_ratio = snoop_pitch / instructor_pitch
semitones_shift = 12 * np.log2(pitch_shift_ratio)
print(f"\n➡️ Required Math Shift: {semitones_shift:.2f} semitones to hit Snoop's deep resonance.")


In [ ]:
# === 3. THE PYTORCH WEIGHT GENERATOR ===
delta_mfcc = torch.tensor(snoop_mfcc - instructor_mfcc, dtype=torch.float32)
print("🧠 PyTorch Sovereign Weight Matrix Calculated:")
print(delta_mfcc)
torch.save(delta_mfcc, 'snoop_dna.pt')
torch.save(torch.tensor([semitones_shift], dtype=torch.float32), 'snoop_pitch_delta.pt')
print("✅ Saved to snoop_dna.pt and snoop_pitch_delta.pt")


In [ ]:
# === 4. PEDALBOARD DSP EXECUTION ===
from pedalboard import Pedalboard, PitchShift, HighpassFilter, LowpassFilter, Compressor, Delay

shift_amount = float(semitones_shift)
shift_amount = max(-12.0, min(12.0, shift_amount))

snoop_board = Pedalboard([
    Compressor(threshold_db=-15, ratio=3),
    PitchShift(semitones=shift_amount),
    LowpassFilter(cutoff_hz=3500),
    Delay(delay_seconds=0.1, mix=0.1)
])

print("🎛️ Executing C++ Pedalboard DSP with calculated Pitch Delta...")
processed_y = snoop_board(instructor_y, sample_rate=sr)
sf.write('snoop_output.wav', processed_y, sr)
print("✅ Voice Conversion Complete!")


In [ ]:
# === 5. AUDIO PLAYBACK VERIFICATION ===
print("🎧 ORIGINAL INSTRUCTOR:")
display(ipd.Audio('instructor_input.m4a'))

print("\n🌿 SNOOP DOGG CONVERTED:")
display(ipd.Audio('snoop_output.wav'))


In [ ]:
# === 6. RAY ACTOR DSP VERIFICATION ===
import ray
import json

print("Connecting to Ray Swarm (Namespace: legion)...")
ray.init(namespace="legion", ignore_reinit_error=True)

@ray.remote
def verify_dsp_footprint(filepath, target_pitch):
    print(f"Worker verifying: {filepath}")
    if not os.path.exists(filepath):
        return {"status": "FAILED", "error": "File not found"}
        
    y, sr = librosa.load(filepath, sr=22050)
    f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    valid_f0 = f0[voiced_flag]
    mean_pitch = np.median(valid_f0) if len(valid_f0) > 0 else 100.0
    
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mean_mfccs = np.mean(mfccs, axis=1)
    
    # Calculate error margins
    pitch_error = abs(mean_pitch - target_pitch)
    
    return {
        "status": "SUCCESS",
        "file": filepath,
        "mean_pitch_hz": float(mean_pitch),
        "target_pitch_hz": float(target_pitch),
        "error_hz": float(pitch_error),
        "mfcc_vector_sum": float(np.sum(mean_mfccs))
    }

print("Dispatching Verification Task to Ray Swarm...")
# Verify the final snoop output matches the target Snoop Pitch (which we calculated earlier)
future = verify_dsp_footprint.remote('snoop_output.wav', float(snoop_pitch))
result = ray.get(future)

print("\n📊 RAY VERIFICATION REPORT:")
print(json.dumps(result, indent=2))

# Save report
with open('ray_verification_report.json', 'w') as f:
    json.dump(result, f, indent=2)
print("✅ Report saved to ray_verification_report.json")


In [ ]:
# === 7. LANGGRAPH DETERMINISTIC OPTIMIZATION LOOP ===
import os
import torch
import librosa
import numpy as np
import soundfile as sf
import ray
from pydantic import BaseModel, Field
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from pedalboard import Pedalboard, Compressor, PitchShift, LowpassFilter

class OptimizerState(TypedDict):
    target_pitch_hz: float
    current_pitch_hz: float
    current_pt_weight: float
    iteration_count: int
    is_optimized: bool
    status_log: list[str]
    last_output_file: str

@ray.remote
class PitchVerifier:
    def verify(self, filepath: str) -> float:
        if not os.path.exists(filepath): return 0.0
        y, sr = librosa.load(filepath, sr=22050)
        f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
        valid_f0 = f0[voiced_flag]
        return float(np.median(valid_f0)) if len(valid_f0) > 0 else 100.0

def node_generate_audio(state: OptimizerState):
    print(f"\n[Iteration {state['iteration_count']}] Generating Audio with Delta: {state['current_pt_weight']:.2f} semitones")
    
    input_path = "instructor_input.m4a"
    if not os.path.exists(input_path):
        print(f"yt-dlp hasn't provided {input_path}, using fallback audio '0 Lead Vocals.wav'...")
        input_path = "0 Lead Vocals.wav"
        if not os.path.exists(input_path):
            return state # Cannot proceed
    
    y, sr = librosa.load(input_path, sr=22050)
    torch.save(torch.tensor([state['current_pt_weight']], dtype=torch.float32), 'snoop_pitch_delta.pt')
    
    board = Pedalboard([
        Compressor(threshold_db=-15, ratio=3),
        PitchShift(semitones=state['current_pt_weight']),
        LowpassFilter(cutoff_frequency_hz=3500)
    ])
    
    processed_y = board(y, sample_rate=sr)
    output_filename = f"snoop_optimized_iter_{state['iteration_count']}.wav"
    sf.write(output_filename, processed_y, sr)
    state["last_output_file"] = output_filename
    state["status_log"].append(f"Generated {output_filename} with delta {state['current_pt_weight']:.2f}")
    return state

def node_verify_audio(state: OptimizerState):
    print("Verifying Pitch via Ray...")
    if not ray.is_initialized(): ray.init(namespace="legion", ignore_reinit_error=True)
    verifier = PitchVerifier.remote()
    actual_pitch = ray.get(verifier.verify.remote(state["last_output_file"]))
    state["current_pitch_hz"] = actual_pitch
    print(f"Result: Target = {state['target_pitch_hz']:.1f}Hz, Actual = {actual_pitch:.1f}Hz")
    
    error = abs(actual_pitch - state['target_pitch_hz'])
    state["is_optimized"] = error < 1.5
    
    if not state["is_optimized"]:
        ratio = state['target_pitch_hz'] / actual_pitch
        correction_semitones = 12 * np.log2(ratio)
        state["current_pt_weight"] += correction_semitones
        print(f"FAILED. Adjusting .pt weight by {correction_semitones:+.2f} semitones.")
    else:
        print("SUCCESS! The .pt file is perfectly optimized.")
        
    state["iteration_count"] += 1
    return state

def pydantic_router(state: OptimizerState):
    if state["is_optimized"] or state["iteration_count"] > 5: return "end"
    return "generate"

# Build Graph
workflow = StateGraph(OptimizerState)
workflow.add_node("generate", node_generate_audio)
workflow.add_node("verify", node_verify_audio)
workflow.add_edge(START, "generate")
workflow.add_edge("generate", "verify")
workflow.add_conditional_edges("verify", pydantic_router, {"generate": "generate", "end": END})
app = workflow.compile()

initial_state = {
    "target_pitch_hz": float(snoop_pitch),
    "current_pitch_hz": 0.0,
    "current_pt_weight": 0.0,
    "iteration_count": 1,
    "is_optimized": False,
    "status_log": [],
    "last_output_file": ""
}

print("Starting Deterministic Optimization Loop...")
final_state = app.invoke(initial_state)
print("OPTIMIZATION COMPLETE! Optimized .pt file saved to: snoop_pitch_delta.pt")


In [ ]:
# === 8. COMPILE NATIVE DESKTOP APPS (Windows & Mac) ===
import subprocess
import os

print("🔨 Compiling Windows Desktop App via PyInstaller...")
try:
    subprocess.run(["python", "-m", "PyInstaller", "--noconsole", "--onefile", "snoop_voice_app.py"], check=True)
    print("✅ Windows App Compiled: Check the /dist folder for snoop_voice_app.exe")
except Exception as e:
    print(f"⚠️ PyInstaller failed: {e}")

print("\n🍏 Generating Mac Build Script (build_mac_app.command)...")
mac_script = """#!/bin/bash
echo "Building Mac Desktop App..."
pip3 install py2app
cat << 'EOF' > setup.py
from setuptools import setup
APP = ['snoop_voice_app.py']
DATA_FILES = ['snoop_dna.pt', 'snoop_pitch_delta.pt']
OPTIONS = {'argv_emulation': True, 'packages': ['torch', 'pedalboard', 'numpy', 'soundfile']}
setup(app=APP, data_files=DATA_FILES, options={'py2app': OPTIONS}, setup_requires=['py2app'])
EOF
python3 setup.py py2app -A
echo "Mac App built in /dist/snoop_voice_app.app!"
"""
with open('build_mac_app.command', 'w') as f:
    f.write(mac_script)
os.chmod('build_mac_app.command', 0o755)
print("✅ Mac Build Script Generated: Run build_mac_app.command on an Apple Silicon device.")
